# media.cptdaz.org (Front Door endpoint cptmtls-media)


Public endpoint without mTLS. Serves `style.css` from a private storage account (Private Link). The stylesheet is referenced by the page on `hello.cptdaz.org`.

Tools: bash, az, http (httpie), jq, azcopy.

## Design notes

* **Domain and certificate:** `media.cptdaz.org` is a CNAME to the endpoint's generated `azurefd.net` hostname, with a Front Door managed certificate after `_dnsauth` validation.
* **Origin:** A private storage account reached through a Front Door private endpoint. It carries the exclusion tag `SecurityControl=Ignore`, so anonymous blob read is not reset by tenant policy. It is readable only behind the private endpoint.
* **Use:** The page on `hello.cptdaz.org` references `https://media.cptdaz.org/style.css`.
* **Approval:** `scripts/approve-pe.sh` approves the pending private endpoint connection after the deployment.
* **Seeding:** `style.css` is written by the VM through its managed identity (Bicep `runCommand`), because shared key access is disabled by policy.


In [ ]:
export SUB=09808f31-065f-4231-914d-776c2d6bbe34
export RG=rg-cptmtls
export DNS_SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f
export DNS_RG=domains
export DNS_ZONE=cptdaz.org
export PROFILE=afd-cptmtls-public
export ENDPOINT=cptmtls-media
export DOMAIN=media-cptdaz-org
export HOST=media.cptdaz.org
export HELLO_HOST=hello.cptdaz.org
export STORAGE=stcptmtlsmediaizhwq
export LAW=law-cptmtls
export BLOB=demo-$(date +%Y%m%d-%H%M%S).css
export PAGER=cat
export AZCOPY_AUTO_LOGIN_TYPE=AZCLI
az account set --subscription $SUB
export LAW_ID=$(az monitor log-analytics workspace show -g $RG -n $LAW --query customerId -o tsv)
env | grep -E '^(SUB|RG|DNS_SUB|DNS_RG|DNS_ZONE|PROFILE|ENDPOINT|DOMAIN|HOST|HELLO_HOST|STORAGE|LAW|BLOB|LAW_ID)=' | sort

## Azure account

In [ ]:
az account show --query "{name:name,id:id,user:user.name}" -o table

## Front Door configuration

In [ ]:
az afd profile show -g $RG --profile-name $PROFILE --query "{name:name,sku:sku.name,state:provisioningState}" -o table

In [ ]:
az afd endpoint show -g $RG --profile-name $PROFILE --endpoint-name $ENDPOINT --query "{host:hostName,enabled:enabledState,deployment:deploymentStatus}" -o table

In [ ]:
az afd custom-domain show -g $RG --profile-name $PROFILE --custom-domain-name $DOMAIN --query "{host:hostName,validation:domainValidationState,provisioning:provisioningState,deployment:deploymentStatus,cert:tlsSettings.certificateType}" -o table

In [ ]:
az afd route list -g $RG --profile-name $PROFILE --endpoint-name $ENDPOINT --query "[].{route:name,patterns:join(',',patternsToMatch),forwarding:forwardingProtocol,deployment:deploymentStatus}" -o table

In [ ]:
az afd origin list -g $RG --profile-name $PROFILE --origin-group-name og-media-storage -o json | jq '.[] | {name, hostName, enabledState, privateLinkTarget: (.sharedPrivateLinkResource.privateLink.id | split("/")[-1])}'

## DNS records in the Azure DNS zone

In [ ]:
az network dns record-set cname show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n media --query "{fqdn:fqdn,target:CNAMERecord.cname}" -o table

In [ ]:
az network dns record-set txt show --subscription $DNS_SUB -g $DNS_RG -z $DNS_ZONE -n _dnsauth.media --query "{fqdn:fqdn,value:TXTRecords[0].value[0]}" -o table

## Private Link connection created by Front Door

In [ ]:
az network private-endpoint-connection list --id $(az storage account show -g $RG -n $STORAGE --query id -o tsv) --query "[].{name:name,status:properties.privateLinkServiceConnectionState.status,description:properties.privateLinkServiceConnectionState.description}" -o table

## Test through Front Door with httpie

In [ ]:
http --headers GET https://$HOST/style.css

In [ ]:
http --body GET https://$HOST/style.css

In [ ]:
http --headers GET https://$HOST/style.css | grep -i content-type

In [ ]:
http --body GET https://$HELLO_HOST/index.html | grep stylesheet

In [ ]:
http --headers GET https://$HOST/missing.css

## Storage origin: az CLI and azcopy over the private endpoint

In [ ]:
getent hosts $STORAGE.blob.core.windows.net

In [ ]:
az storage blob list --account-name $STORAGE --container-name '$root' --auth-mode login -o json | jq '.[] | {name, size: .properties.contentLength, type: .properties.contentSettings.contentType}'

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/\$root"

In [ ]:
azcopy copy "https://$STORAGE.blob.core.windows.net/\$root/style.css" /tmp/style-azcopy.css

In [ ]:
http --body GET https://$HOST/style.css | diff - /tmp/style-azcopy.css && echo identical

## Publish a new file with azcopy and read it through Front Door

In [ ]:
cp /tmp/style-azcopy.css /tmp/$BLOB

In [ ]:
azcopy copy /tmp/$BLOB "https://$STORAGE.blob.core.windows.net/\$root/$BLOB"

In [ ]:
http --headers GET https://$HOST/$BLOB

In [ ]:
az storage blob delete --account-name $STORAGE --container-name '$root' --name $BLOB --auth-mode login

In [ ]:
azcopy list "https://$STORAGE.blob.core.windows.net/\$root"

## Front Door access logs in Log Analytics

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | summarize events=count() by Category" -o json | jq -c '.[]'

In [ ]:
az monitor log-analytics query -w $LAW_ID --analytics-query "AzureDiagnostics | where ResourceProvider == 'MICROSOFT.CDN' | where ResourceId contains toupper('$PROFILE') | top 5 by TimeGenerated desc | project TimeGenerated, Category, ResourceId" -o json | jq -c '.[]'